In [71]:
import numpy as np
import pandas as pd
import torch
import re
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from huggingface_hub import notebook_login
from tqdm.auto import tqdm

In [4]:
#I. Обучить модель, которая будет по prompt и response предсказывать, какая модель была использована
#    1. Объединить в один датасет все prompt и response
#    2. Удалить мусорные символы из prompt и response
#    3. Разделим выборку на тестовую и обучающую
#    4. Балансировка классов:
#        4.1. Увеличить объем миноритарных классов добавлением данных со стороннего ресурса
#        4.2. Сгруппируем близкие версии одной модели в одно семейство
#        4.3. Back_Translation:
#            4.3.1. Для миноритарных классов определить язык запроса и язык ответа
#            4.3.2. Для миноритарных классов оставить текста на английском языке, длиной более 50 символов, с минимальным
#            кол-вом специальных символов
#            4.3.3. Разбить текста по предложениям, оставить предложения с длиной [20,1500] символов
#            4.3.4. Увеличить объем миноритарных классов путем обратного перевода запроса и ответа (Переводить по предложениям)


#        3.1. Из каждого мажоритарного класса удалить часть данных
#(llama-2-7b-chat, guanaco-33b, stablelm-tuned-alpha-7b, dolly-v2-12b)

#    5. Можно добавить в качестве признаков длину запроса, длину ответа, язык запроса, язык ответа, совпадают ли язык запроса
#       и язык ответа, количество символов {`,**,#}, наличие паттернов

In [2]:
notebook_login()

In [3]:
#Загрузка исходных данных
origin_data = pd.read_csv('train.csv')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [37]:
#Загрузка моделей (локально)
#Определение языка текста
tokenizer_lang = AutoTokenizer.from_pretrained('papluca/xlm-roberta-base-language-detection')
model_lang = AutoModelForSequenceClassification.from_pretrained('papluca/xlm-roberta-base-language-detection')
print('papluca/xlm-roberta-base-language-detection готова')
tokenizer_en_de = AutoTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-de')
model_en_de = AutoModelForSeq2SeqLM.from_pretrained('Helsinki-NLP/opus-mt-en-de')
print('Helsinki-NLP/opus-mt-en-de готова')
tokenizer_de_en = AutoTokenizer.from_pretrained('Helsinki-NLP/opus-mt-de-en')
model_de_en = AutoModelForSeq2SeqLM.from_pretrained('Helsinki-NLP/opus-mt-de-en')
print('Helsinki-NLP/opus-mt-de-en готова')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

papluca/xlm-roberta-base-language-detection готова


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Helsinki-NLP/opus-mt-en-de готова


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Helsinki-NLP/opus-mt-de-en готова


In [142]:
#Методы
def getNewData(file_name, model_types):
    add_data = pd.read_parquet(file_name)
    add_data = add_data[(add_data['model'].isin(model_types)) 
                        & (add_data['language'] == 'English')]
    add_data['prompt'] = add_data['conversation'].apply(lambda arr: arr[0]['content'])
    add_data['response'] = add_data['conversation'].apply(lambda arr: arr[1]['content'])
    print(f'Выполнено добавление данных из файла {file_name}')
    return pd.DataFrame(add_data[['prompt','response','model']])

def getFamilyName(strings):
    try:
        return f'{strings[0]}-{float(strings[1])}'
    except ValueError:
        return strings[0]

def getTextLang(model, 
                tokenizer,
                max_length,
                dataset,
                batch_size,
                device):
    if type(dataset) != list:
        raise TypeError('Массив текстов должен быть типа List')
    model.to(device)
    model.eval()
    result = []
    with torch.no_grad():
        for index in tqdm(range(0, len(dataset), batch_size)):
            batch = dataset[index: index+batch_size]
            inputs = tokenizer(batch,
                               return_tensors='pt',
                               padding=True, 
                               truncation=True, 
                               max_length=max_length)
            inputs = {param: value.to(device) for param, value in inputs.items()}
            logits = model(**inputs).logits.cpu()
            result.extend([model.config.id2label[lang_id.item()] 
                           for lang_id in torch.argmax(torch.softmax(logits, dim=1), dim=1)])
    return result

def translate(model, 
              tokenizer,
              dataset,
              device):
    if type(dataset) != list:
        raise TypeError('Массив текстов должен быть типа List')
    model.to(device)
    model.eval()
    result = []
    with torch.no_grad():
        for batch in tqdm(dataset):
            inputs = tokenizer(batch, return_tensors='pt', padding=True)
            inputs = {param: value.to(device) for param, value in inputs.items()}
            outputs = model.generate(**inputs)
            outputs = tokenizer.decode(outputs, skip_special_tokens=True)
            result.append(outputs)
    return result



In [6]:
#I-1
data = pd.DataFrame(np.vstack((origin_data[['prompt','response_a','model_a']],
                               origin_data[['prompt','response_b','model_b']])),
                    columns=['prompt','response','model'])

#I-2
data['prompt'] = data['prompt'].str[2:-2]
data['response'] = data['response'].str[2:-2]

#I-3
data_training, data_testing = train_test_split(data,
                                              test_size=0.2,
                                              random_state=101,
                                              stratify=data['model']) 

#I-4.1
for file_name in ['0000.parquet','0001.parquet']:
    add_data = getNewData(file_name, ['llama-13b','guanaco-33b','stablelm-tuned-alpha-7b','dolly-v2-12b'])
    data_training = pd.concat([data_training, add_data], ignore_index=True)

#I-4.2
data_training[data_training['model'] == 'claude-instant-1'] = 'claude-1'
data_training[data_training['model'] == 'claude-2.1'] = 'claude-2.0'
data_training[data_training['model'] == 'gpt4all-13b-snoozy'] = 'gpt-4-0314'
data_training[data_training['model'] == 'llama2-70b-steerlm-chat'] = 'llama-2-7b-chat'
data_training[data_training['model'] == 'chatglm2-6b'] = 'chatglm3-6b'
data_training[data_training['model'] == 'qwen-14b-chat'] = 'qwen1.5-72b-chat'
data_training['model'] = data_training['model'].str.split('-').apply(getFamilyName)

#I-4.3.1
minor_classes = data_training['model'].value_counts().nsmallest(10).index
data_minor = data_training[data_training['model'].isin(minor_classes)].copy()
data_minor['prompt_lang'] = getTextLang(model_lang, tokenizer_lang, 80, data_minor['prompt'].to_list(), 512, device)
data_minor['response_lang'] = getTextLang(model_lang, tokenizer_lang, 80, data_minor['response'].to_list(), 512, device)

#I-4.3.2
data_minor = data_minor[(data_minor['prompt_lang'] == 'en') 
                        & (data_minor['response_lang'] == 'en')
                        & (data_minor['prompt'].str.len() >= 50)
                        & (data_minor['response'].str.len() >= 50)
                        & (data_minor['response'].str.count(r'\\|#|\*') <= 50)].copy()

#I-4.3.3
data_minor['response_seq'] = data_minor['response'].str.split(r'\\n|\.|\n', 
                                                              regex=True).apply(lambda arr: [seq for seq in arr 
                                                                                             if 20 <= len(seq) <= 1500])
data_minor['prompt_seq'] = data_minor['prompt'].str.split(r'\\n|\.|\n', 
                                                          regex=True).apply(lambda arr: [seq for seq in arr 
                                                                                         if 20 <= len(seq) <= 1500])

Выполнено добавление данных из файла 0000.parquet
Выполнено добавление данных из файла 0001.parquet


In [ ]:
def translate(model, 
              tokenizer,
              dataset,
              device):
    if type(dataset) != list:
        raise TypeError('Массив текстов должен быть типа List')
    model.to(device)
    model.eval()
    result = []
    with torch.no_grad():
        for batch in tqdm(dataset):
            inputs = tokenizer(batch, return_tensors='pt', padding=True)
            inputs = {param: value.to(device) for param, value in inputs.items()}
            outputs = model.generate(**inputs)
            outputs = tokenizer.decode(outputs, skip_special_tokens=True)
            result.append(outputs)
    return result

In [161]:
def backTranslation(f_model, 
                    f_tokenizer, 
                    b_model, 
                    b_tokenizer, 
                    dataset, 
                    device):
    f_response_seq = translate(f_model, f_tokenizer, dataset['response_seq'].to_list(), device)
    b_response_seq = translate(b_model, b_tokenizer, f_response_seq, device)
    f_prompt_seq = translate(f_model, f_tokenizer, dataset['prompt_seq'].to_list(), device)
    b_prompt_seq = translate(b_model, b_tokenizer, f_prompt_seq, device)
    result = {'prompt': list(map(lambda seq: '.'.join(seq), b_prompt_seq)),
              'response': list(map(lambda seq: '.'.join(seq), b_response_seq)),
              'model': dataset['model'].to_list()}
    return pd.DataFrame(result)

In [162]:
d = backTranslation(model_en_de,
                    tokenizer_en_de,
                    model_de_en,
                    tokenizer_de_en,
                    data_minor.iloc[:3],
                    device)

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]